# 5. Programs

Standalone notebook: it includes its own imports and runs top to bottom with no input prompts. Each `assert` raises an error if something is wrong, so no error means the section works.

In [ ]:
import pandas as pd

## Simple invoice generator

The invoice calculation takes a `{item: quantity}` dictionary, so it needs no prompts.
`ask_quantity` keeps the original input validation for interactive use.

In [ ]:
PRICE_LIST = {
    "laptop": {"name": "Laptop", "price": 55000},
    "mouse": {"name": "Mouse", "price": 800},
    "speakers": {"name": "Speakers", "price": 2500},
    "bluetooth": {"name": "Bluetooth Device", "price": 1500},
}


def ask_quantity(item_key: str, input_fn=input) -> int:
    """Ask for a quantity until the user enters a whole number."""
    item = PRICE_LIST[item_key]
    print(f"\n{item['name']} - Price per unit: Rs {item['price']}")
    while True:
        answer = input_fn(f"How many {item['name']}(s) do you want to buy? (0 if none): ")
        if answer.isdigit():
            return int(answer)
        print("Please enter a valid whole number.")


def collect_quantities(input_fn=input) -> dict:
    return {key: ask_quantity(key, input_fn) for key in PRICE_LIST}


def build_invoice(quantities: dict):
    """Return (invoice_lines, grand_total) for the quantities given."""
    lines, grand_total = [], 0
    for key, item in PRICE_LIST.items():
        qty = quantities.get(key, 0)
        if qty > 0:
            line_total = qty * item["price"]
            grand_total += line_total
            lines.append({"name": item["name"], "qty": qty,
                          "unit_price": item["price"], "line_total": line_total})
    return lines, grand_total


def print_invoice(lines: list, grand_total: int) -> None:
    print("\n" + "=" * 50)
    print(" " * 15 + "FINAL INVOICE")
    print("=" * 50)
    if not lines:
        print("No items were purchased.")
        return
    print(f"{'Item':<18}{'Qty':<6}{'Unit Price':<14}{'Line Total':<12}")
    print("-" * 50)
    for line in lines:
        print(f"{line['name']:<18}{line['qty']:<6}{line['unit_price']:<14}{line['line_total']:<12}")
    print("-" * 50)
    print(f"{'GRAND TOTAL:':<38}Rs {grand_total}")
    print("=" * 50)


# Interactive version: print_invoice(*build_invoice(collect_quantities()))
lines, total = build_invoice({"laptop": 1, "mouse": 2, "bluetooth": 1})
print_invoice(lines, total)

assert total == 55000 + 2 * 800 + 1500
assert build_invoice({}) == ([], 0)

answers = iter(["abc", "-1", "2"])   # two invalid answers, then a valid one
assert ask_quantity("mouse", lambda prompt: next(answers)) == 2

## Student information

In [ ]:
STUDENT_NAMES = ["Babar", "Usman", "Ali"]
STUDENT_AGES = [20, 22, 21]
STUDENT_GRADES = [85, 90, 78]

student_info = pd.DataFrame({
    "name": STUDENT_NAMES,
    "age": STUDENT_AGES,
    "grade": STUDENT_GRADES,
})
student_info

## Student grading system

A `GradingSystem` class holds the logic and raises `ValueError` for bad input. The menu is a
separate function, so the same code runs interactively or from scripted answers.

In [ ]:
class GradingSystem:
    GRADE_BANDS = [(90, "A+"), (80, "A"), (70, "B"), (60, "C"), (50, "D")]

    def __init__(self):
        self.students = {}

    def _get(self, roll_number: str) -> dict:
        if roll_number not in self.students:
            raise ValueError("Student not found.")
        return self.students[roll_number]

    def add_student(self, roll_number: str, name: str) -> None:
        if roll_number in self.students:
            raise ValueError("A student with this roll number already exists.")
        self.students[roll_number] = {"name": name, "marks": {}}

    def add_marks(self, roll_number: str, subject: str, marks: float) -> None:
        student = self._get(roll_number)
        if not 0 <= marks <= 100:
            raise ValueError("Marks must be between 0 and 100.")
        student["marks"][subject] = marks

    def average(self, roll_number: str) -> float:
        marks = self._get(roll_number)["marks"]
        return sum(marks.values()) / len(marks) if marks else 0

    @classmethod
    def grade_for(cls, average: float) -> str:
        for threshold, grade in cls.GRADE_BANDS:
            if average >= threshold:
                return grade
        return "F"

    def report(self, roll_number: str) -> str:
        student = self._get(roll_number)
        line = "-" * 40
        out = [line, "Student Report Card", line,
               f"Roll Number: {roll_number}", f"Name: {student['name']}"]
        if not student["marks"]:
            out.append("No marks recorded yet.")
            return "\n".join(out)
        out.append("\nSubject-wise Marks:")
        out += [f" - {subject} : {marks}" for subject, marks in student["marks"].items()]
        average = self.average(roll_number)
        out += [f"\nAverage Marks: {round(average, 2)}",
                f"Final Grade: {self.grade_for(average)}", line]
        return "\n".join(out)

    def summary(self) -> str:
        if not self.students:
            return "No students found."
        line = "-" * 40
        rows = [line, "All Students Summary", line]
        for roll, student in self.students.items():
            average = self.average(roll)
            rows.append(f"Roll No: {roll} | Name: {student['name']} | "
                        f"Average: {round(average, 2)} | Grade: {self.grade_for(average)}")
        rows.append(line)
        return "\n".join(rows)


GRADING_MENU = """
========== STUDENT GRADING SYSTEM ==========
1. Add New Student
2. Add / Update Marks
3. View Student Report
4. View All Students
5. Exit
=============================================="""


def run_grading_menu(system: GradingSystem, input_fn=input) -> None:
    while True:
        print(GRADING_MENU)
        choice = input_fn("Enter your choice (1-5): ").strip()
        try:
            if choice == "1":
                roll = input_fn("Enter student roll number: ")
                system.add_student(roll, input_fn("Enter student name: "))
                print("Student added successfully!")
            elif choice == "2":
                roll = input_fn("Enter student roll number: ")
                subject = input_fn("Enter subject name: ")
                system.add_marks(roll, subject, float(input_fn("Enter marks obtained (out of 100): ")))
                print("Marks recorded successfully!")
            elif choice == "3":
                print(system.report(input_fn("Enter student roll number: ")))
            elif choice == "4":
                print(system.summary())
            elif choice == "5":
                print("Thank you!")
                break
            else:
                print("Invalid choice. Please enter a number between 1 and 5.")
        except ValueError as error:
            print("Error:", error)


# Interactive version: run_grading_menu(GradingSystem())
school = GradingSystem()
script = iter(["1", "101", "Ali",
               "2", "101", "Math", "95",
               "2", "101", "Physics", "85",
               "2", "101", "Chemistry", "150",   # rejected: out of range
               "3", "101",
               "4",
               "5"])
run_grading_menu(school, lambda prompt: next(script))

assert school.average("101") == 90
assert GradingSystem.grade_for(90) == "A+" and GradingSystem.grade_for(49) == "F"

## Banking system

Same structure as the grading system: a `Bank` class with the rules, and a menu function around it.

In [ ]:
class Bank:
    def __init__(self, first_account_number: int = 1001):
        self.accounts = {}
        self._next_number = first_account_number

    def _get(self, number: int) -> dict:
        if number not in self.accounts:
            raise ValueError("Account not found.")
        return self.accounts[number]

    @staticmethod
    def _check_amount(amount: float) -> None:
        if amount <= 0:
            raise ValueError("Amount must be positive.")

    def create_account(self, name: str, starting_balance: float) -> int:
        if starting_balance < 0:
            raise ValueError("Starting deposit cannot be negative.")
        number = self._next_number
        self._next_number += 1
        self.accounts[number] = {"name": name, "balance": starting_balance}
        return number

    def deposit(self, number: int, amount: float) -> float:
        account = self._get(number)
        self._check_amount(amount)
        account["balance"] += amount
        return account["balance"]

    def withdraw(self, number: int, amount: float) -> float:
        account = self._get(number)
        self._check_amount(amount)
        if amount > account["balance"]:
            raise ValueError("Insufficient balance.")
        account["balance"] -= amount
        return account["balance"]

    def balance(self, number: int) -> float:
        return self._get(number)["balance"]

    def transfer(self, sender: int, receiver: int, amount: float) -> float:
        """Move money between accounts; returns the sender's new balance."""
        self._get(sender)
        self._get(receiver)
        self._check_amount(amount)
        self.withdraw(sender, amount)
        self.deposit(receiver, amount)
        return self.balance(sender)


BANK_MENU = """
========== BANKING SYSTEM ==========
1. Create New Account
2. Deposit Money
3. Withdraw Money
4. Check Balance
5. Transfer Money
6. View All Accounts
7. Exit
====================================="""


def run_bank_menu(bank: Bank, input_fn=input) -> None:
    while True:
        print(BANK_MENU)
        choice = input_fn("Enter your choice (1-7): ").strip()
        try:
            if choice == "1":
                name = input_fn("Enter account holder name: ")
                number = bank.create_account(name, float(input_fn("Enter starting deposit amount: ")))
                print(f"Account created successfully! Your account number is: {number}")
            elif choice == "2":
                number = int(input_fn("Enter account number: "))
                print("Deposit successful! New balance:",
                      bank.deposit(number, float(input_fn("Enter amount to deposit: "))))
            elif choice == "3":
                number = int(input_fn("Enter account number: "))
                print("Withdrawal successful! New balance:",
                      bank.withdraw(number, float(input_fn("Enter amount to withdraw: "))))
            elif choice == "4":
                number = int(input_fn("Enter account number: "))
                print("Account Holder:", bank.accounts.get(number, {}).get("name", "?"))
                print("Current Balance:", bank.balance(number))
            elif choice == "5":
                sender = int(input_fn("Enter your account number: "))
                receiver = int(input_fn("Enter receiver's account number: "))
                print("Transfer successful! Your new balance:",
                      bank.transfer(sender, receiver, float(input_fn("Enter amount to transfer: "))))
            elif choice == "6":
                if not bank.accounts:
                    print("No accounts found.")
                for number, acct in bank.accounts.items():
                    print(f"Account Number: {number} | Name: {acct['name']} | Balance: {acct['balance']}")
            elif choice == "7":
                print("Thank you for using the Banking System. Goodbye!")
                break
            else:
                print("Invalid choice. Please enter a number between 1 and 7.")
        except ValueError as error:
            print("Error:", error)


# Interactive version: run_bank_menu(Bank())
bank = Bank()
script = iter(["1", "Ali", "5000",
               "1", "Sara", "1000",
               "2", "1001", "500",
               "3", "1002", "5000",       # rejected: insufficient balance
               "5", "1001", "1002", "2000",
               "4", "1002",
               "6",
               "7"])
run_bank_menu(bank, lambda prompt: next(script))

assert bank.balance(1001) == 3500
assert bank.balance(1002) == 3000

try:
    bank.withdraw(1001, 10**9)
except ValueError as error:
    assert str(error) == "Insufficient balance."
else:
    raise AssertionError("Expected an overdraft to be rejected")

In [ ]:
print("Programs: all tests passed.")